# UFLD v1 TuSimple ONNX evaluation on Raspberry Pi 5

Run this notebook inside the supplied ARM64 Docker container. It measures accuracy, CPU latency and throughput, memory use, CPU utilization, temperature/frequency, ONNX operator timings, and an estimated model tensor-I/O volume. Energy requires a measured power input.

Expected model: exported FP32 ONNX from the baseline notebook, input `[N,3,288,800]`, output `[N,101,56,4]`.

In [ ]:
from pathlib import Path
import os, json, time, threading, platform, subprocess
import numpy as np, pandas as pd, psutil, onnx, onnxruntime as ort
from PIL import Image
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

MODEL = Path('/models/ufld_v1_tusimple.onnx')
GT_FILE = Path('/data/test_label.json')
IMAGE_ROOT = Path('/data')  # change to /data/test_set when raw_file paths live there
RESULTS = Path('/results'); RESULTS.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get('OMP_NUM_THREADS', '4'))
MAX_IMAGES = None  # set e.g. 200 for a quick pass; None uses every label
WARMUP, REPEATS = 20, 100
assert MODEL.is_file(), MODEL
assert GT_FILE.is_file(), GT_FILE
print(platform.platform(), '| Python', platform.python_version(), '| ORT', ort.__version__, '| threads', THREADS)
print('model MB:', round(MODEL.stat().st_size / 2**20, 2))

In [ ]:
# Runtime session and model metadata. Enable ORT's built-in per-node profiling.
so = ort.SessionOptions()
so.intra_op_num_threads = THREADS
so.inter_op_num_threads = 1
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
so.enable_profiling = True
so.profile_file_prefix = str(RESULTS / 'ort_profile')
session = ort.InferenceSession(str(MODEL), sess_options=so, providers=['CPUExecutionProvider'])
inp = session.get_inputs()[0].name; out = session.get_outputs()[0].name
print('input:', session.get_inputs()[0].shape, '| output:', session.get_outputs()[0].shape)
assert session.get_providers()[0] == 'CPUExecutionProvider'
def preprocess(path):
    im = Image.open(path).convert('RGB').resize((800, 288), Image.Resampling.BILINEAR)
    a = np.asarray(im, dtype=np.float32) / 255.0
    a = (a - np.array([.485,.456,.406],np.float32)) / np.array([.229,.224,.225],np.float32)
    return np.transpose(a, (2,0,1))[None].copy()
def infer(x): return session.run([out], {inp: x})[0]
x0 = np.zeros((1,3,288,800), np.float32)
for _ in range(WARMUP): infer(x0)
print('warm-up complete')

In [ ]:
# Single-image model latency and sampled process/container utilization.
proc = psutil.Process(); samples=[]; stop=threading.Event()
def monitor():
    while not stop.is_set():
        try:
            m=proc.memory_info()
            samples.append({'t':time.time(),'rss_mb':m.rss/2**20,'cpu_pct':proc.cpu_percent(None),
                            'system_cpu_pct':psutil.cpu_percent(None)})
        except Exception: pass
        stop.wait(.2)
proc.cpu_percent(None); psutil.cpu_percent(None)
mon=threading.Thread(target=monitor,daemon=True); mon.start()
lat=[]
for _ in range(REPEATS):
    t=time.perf_counter(); infer(x0); lat.append((time.perf_counter()-t)*1000)
stop.set(); mon.join()
lat=np.asarray(lat)
latency={'mean_ms':float(lat.mean()),'std_ms':float(lat.std()),'p50_ms':float(np.percentile(lat,50)),
         'p95_ms':float(np.percentile(lat,95)),'p99_ms':float(np.percentile(lat,99)),
         'throughput_images_s':float(1000/lat.mean()),'threads':THREADS}
pd.Series(latency).to_frame('value')

In [ ]:
# cgroup memory limit/current usage (when available), CPU and Pi thermal/frequency readings.
def read_text(p):
    try: return Path(p).read_text().strip()
    except Exception: return None
cgroup={p:read_text(p) for p in ['/sys/fs/cgroup/memory.current','/sys/fs/cgroup/memory.max',
                                   '/sys/fs/cgroup/cpu.stat']}
thermal=[]
for p in Path('/sys/class/thermal').glob('thermal_zone*/temp'):
    v=read_text(p)
    if v and v.isdigit(): thermal.append(int(v)/1000)
freq=[]
for p in Path('/sys/devices/system/cpu').glob('cpu[0-9]*/cpufreq/scaling_cur_freq'):
    v=read_text(p)
    if v and v.isdigit(): freq.append(int(v)/1000)
util=pd.DataFrame(samples)
print('process max RSS MB:', round(util.rss_mb.max(),1) if len(util) else None)
print('sampled process CPU %:', round(util.cpu_pct.mean(),1) if len(util) else None)
print('sampled system CPU %:', round(util.system_cpu_pct.mean(),1) if len(util) else None)
print('cgroup:', cgroup)
print('thermal °C:', thermal, '| cpu freq MHz:', freq)

In [ ]:
# TuSimple post-processing matching the baseline decoder.
GRID, NROWS, NLANES = 100, 56, 4
H_SAMPLES = list(range(160,720,10))
def decode(logits):
    bg=np.argmax(logits,axis=0)==GRID
    z=logits[:-1]-np.max(logits[:-1],axis=0,keepdims=True)
    q=np.exp(z); q/=q.sum(axis=0,keepdims=True)
    pos=(q*np.arange(GRID,dtype=np.float32)[:,None,None]).sum(axis=0); pos[bg]=GRID
    lanes=[]
    for lane in range(NLANES):
        lanes.append([-2 if pos[r,lane]==GRID else int(round((pos[r,lane]+.5)*1280/(GRID-1))) for r in range(NROWS)])
    return lanes
def lane_match(pred, gt):
    valid=np.asarray(gt)>=0
    if valid.sum()==0: return False
    return float(np.mean(np.abs(np.asarray(pred)[valid]-np.asarray(gt)[valid])<20)) > .85
def evaluate_pair(pred_lanes, gt_lanes):
    if not pred_lanes or not gt_lanes: return 0, len(pred_lanes), len(gt_lanes)
    cost=np.ones((len(gt_lanes),len(pred_lanes)),dtype=float)
    for i,g in enumerate(gt_lanes):
        for j,p in enumerate(pred_lanes): cost[i,j]=0 if lane_match(p,g) else 1
    ri,ci=linear_sum_assignment(cost)
    matched=int(sum(cost[i,j]==0 for i,j in zip(ri,ci)))
    return matched, len(pred_lanes)-matched, len(gt_lanes)-matched

In [ ]:
# Full labeled test-set accuracy and end-to-end timing.
records=[json.loads(s) for s in GT_FILE.read_text().splitlines() if s.strip()]
if MAX_IMAGES: records=records[:MAX_IMAGES]
pred_lines=[]; match_total=fp_total=fn_total=gt_total=0; e2e_ms=[]; model_ms=[]
for k,rec in enumerate(records):
    image_path=IMAGE_ROOT / rec['raw_file']
    if not image_path.is_file(): raise FileNotFoundError(image_path)
    t=time.perf_counter(); x=preprocess(image_path); t1=time.perf_counter(); logits=infer(x)[0]; t2=time.perf_counter()
    lanes=decode(logits); gt_lanes=rec['lanes']
    m,fp,fn=evaluate_pair(lanes,gt_lanes)
    match_total+=m; fp_total+=fp; fn_total+=fn; gt_total+=len(gt_lanes)
    model_ms.append((t2-t1)*1000); e2e_ms.append((t2-t)*1000)
    pred_lines.append({'lanes':lanes,'h_samples':rec.get('h_samples',H_SAMPLES),'raw_file':rec['raw_file'],'run_time':10})
    if (k+1)%100==0: print(k+1,'/',len(records))
(RESULTS/'predictions.jsonl').write_text(''.join(json.dumps(x)+'\n' for x in pred_lines))
accuracy=match_total/max(gt_total,1); fp_rate=fp_total/max(len(records),1); fn_rate=fn_total/max(len(records),1)
eval_summary={'images':len(records),'Accuracy':accuracy,'FP':fp_rate,'FN':fn_rate,
 'model_mean_ms':float(np.mean(model_ms)),'model_p95_ms':float(np.percentile(model_ms,95)),
 'end_to_end_mean_ms':float(np.mean(e2e_ms)),'end_to_end_p95_ms':float(np.percentile(e2e_ms,95)),
 'end_to_end_throughput_images_s':float(1000/np.mean(e2e_ms))}
pd.Series(eval_summary).to_frame('value')

In [ ]:
# Per-node ONNX Runtime profile. ORT records node duration in microseconds.
profile_path=session.end_profiling(); print('profile:',profile_path)
events=json.loads(Path(profile_path).read_text())
node_rows=[]
for e in events:
    if e.get('cat')=='Node' and e.get('ph')=='X':
        node_rows.append({'name':e.get('name'),'op':e.get('args',{}).get('op_name'),
                          'provider':e.get('args',{}).get('provider'),'duration_ms':e.get('dur',0)/1000})
node_df=pd.DataFrame(node_rows)
if len(node_df):
    display(node_df.groupby(['op','provider'],dropna=False).duration_ms.agg(['count','sum','mean']).sort_values('sum',ascending=False).head(30))
    node_df.to_csv(RESULTS/'operator_profile.csv',index=False)
    display(node_df.nlargest(25,'duration_ms')[['op','duration_ms','name']])
else: print('No node events found; inspect the profile JSON format for this ORT version.')

In [ ]:
# Static estimate of node tensor I/O and initializers (logical byte volume, not DRAM traffic).
graph=onnx.shape_inference.infer_shapes(onnx.load(str(MODEL))).graph
shapes={}
def dims(value_info):
    tt=value_info.type.tensor_type
    if not tt.HasField('shape'): return None
    ds=[]
    for d in tt.shape.dim:
        if d.HasField('dim_value'): ds.append(int(d.dim_value))
        else: return None
    return ds
for vi in list(graph.input)+list(graph.value_info)+list(graph.output): shapes[vi.name]=dims(vi)
init_bytes={x.name:len(x.raw_data) if x.raw_data else int(np.prod(x.dims))*4 for x in graph.initializer}
tensor_bytes={name:int(np.prod(shape))*4 for name,shape in shapes.items() if shape is not None}
rows=[]
for node in graph.node:
    ib=sum(init_bytes.get(n,tensor_bytes.get(n,0)) for n in node.input)
    ob=sum(tensor_bytes.get(n,0) for n in node.output)
    rows.append({'name':node.name or node.op_type,'op':node.op_type,'input_output_MB':(ib+ob)/2**20})
traffic=pd.DataFrame(rows).sort_values('input_output_MB',ascending=False)
estimated_mb=float(traffic.input_output_MB.sum())
print('Estimated aggregate node tensor I/O MB/inference:',round(estimated_mb,1))
print('Estimate may include unknown shapes as zero and counts reused tensor reads/writes repeatedly.')
traffic.head(25)

In [ ]:
# Energy: enter readings from a USB-C power meter, taken at the same point.
# Keep None if unavailable. Gross = average watts × measured model seconds/image.
# Idle-subtracted power = loaded watts - idle watts.
AVG_SYSTEM_WATTS = None  # e.g. 7.2
IDLE_SYSTEM_WATTS = None # e.g. 3.4
energy={}
if AVG_SYSTEM_WATTS is not None:
    energy['gross_J_per_inference']=AVG_SYSTEM_WATTS*(eval_summary['model_mean_ms']/1000)
    if IDLE_SYSTEM_WATTS is not None:
        energy['above_idle_J_per_inference']=(AVG_SYSTEM_WATTS-IDLE_SYSTEM_WATTS)*(eval_summary['model_mean_ms']/1000)
print(energy or 'No power readings: energy not calculated')

In [ ]:
# Save a reproducible summary and key measurements.
summary={'model_path':str(MODEL),'model_size_MB':MODEL.stat().st_size/2**20,
 'runtime':'onnxruntime CPUExecutionProvider','ort_version':ort.__version__,'threads':THREADS,
 'latency_fixed_batch1':latency,'dataset_evaluation':eval_summary,
 'process_rss_max_MB':float(util.rss_mb.max()) if len(util) else None,
 'process_cpu_avg_percent':float(util.cpu_pct.mean()) if len(util) else None,
 'system_cpu_avg_percent':float(util.system_cpu_pct.mean()) if len(util) else None,
 'cgroup':cgroup,'temperature_C':thermal,'cpu_frequency_MHz':freq,
 'estimated_node_tensor_io_MB_per_inference':estimated_mb,'energy':energy,
 'energy_scope':'whole-board meter input' if energy else None}
(RESULTS/'summary.json').write_text(json.dumps(summary,indent=2))
pd.Series({'model_MB':summary['model_size_MB'],'latency_p95_ms':latency['p95_ms'],
 'throughput_img_s':latency['throughput_images_s'],'max_RSS_MB':summary['process_rss_max_MB'],
 'estimated_tensor_IO_MB':estimated_mb,'accuracy':accuracy,'FP':fp_rate,'FN':fn_rate}).to_frame('value')